##### Copyright 2026 The MediaPipe Authors. All Rights Reserved.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Decision Maker with MediaPipe Tasks

This notebook shows you how to use the MediaPipe Tasks Python API to answer structured questions about a piece of text with calibrated probabilities:

- **Boolean**: does a statement hold for the text? (yes / no)
- **Choice**: which of several categories fits the text best?
- **Score**: where does the text sit on an ordered rubric (for example 1 to 5)?

## Preparation

Let's start with installing MediaPipe.

In [ ]:
!pip install mediapipe

Then download an off-the-shelf model. This example uses [EmbeddingGemma 2 270M](https://huggingface.co/litert-community/embeddinggemma-2-text-270m-litert-lm) (~165 MB) by default. You can also pick Laya S256 (~680 MB), a small encoder model for on-device decisions.

In [ ]:
import urllib.request

model = "EmbeddingGemma 2 270M" #@param ["EmbeddingGemma 2 270M", "Laya S256"]

MODEL_URLS = {
    'EmbeddingGemma 2 270M': 'https://huggingface.co/litert-community/embeddinggemma-2-text-270m-litert-lm/resolve/main/embeddinggemma-2-text-270m.litertlm',
    'Laya S256': 'https://storage.googleapis.com/mediapipe-models/decision_maker/laya/float32/laya_s256/latest/laya_s256.task',
}
model_url = MODEL_URLS[model]
model_path = model_url.split('/')[-1]
urllib.request.urlretrieve(model_url, model_path)
print(f'Downloaded {model} to {model_path}')

## Running inference

To run inference, initialize the `DecisionMaker` with the model that you downloaded earlier, then evaluate a question against your input text. You can edit the text and the question on the side of each section.

### Boolean question

A `BooleanQuestion` checks whether a condition holds for the text. The result includes `value` (the yes / no decision for the given `threshold`) and `probability_true`.

In [ ]:
from mediapipe.tasks import python
from mediapipe.tasks.python import decision

# Create the DecisionMaker with the model that was downloaded earlier.
base_options = python.BaseOptions(model_asset_path=model_path)
options = decision.DecisionMakerOptions(base_options=base_options)
decision_maker = decision.DecisionMaker.create_from_options(options)

input_text = "This product is amazing, it works perfectly and I use it every day!" #@param {type:"string"}
condition = "The review is positive." #@param {type:"string"}

result = decision_maker.evaluate_boolean(
    input_text, decision.BooleanQuestion(condition=condition, threshold=0.5))

print(f'Answer: {"YES" if result.value else "NO"}')
print(f'P(true): {result.probability_true:.1%}')

### Choice question

A `ChoiceQuestion` picks the best category. Describe each category in `criteria`; the result includes the `selected_key` and a probability per category.

In [ ]:
input_text = "My package never arrived even though tracking says delivered." #@param {type:"string"}

choice_question = decision.ChoiceQuestion(
    criteria={
        'shipping': 'Delivery problems, lost packages, tracking',
        'billing': 'Payment issues, duplicate charges, invoices',
        'technical': 'App crashes, login errors, bugs',
    },
    instructions='Which department should handle this ticket?',
)
result = decision_maker.evaluate_choice(input_text, choice_question)

print(f'Selected: {result.selected_key}')
for key, probability in sorted(result.probabilities.items(), key=lambda kv: -kv[1]):
  print(f'  {key:10s} {probability:.1%}')

### Score question

A `ScoreQuestion` rates the text on an ordered `rubric`, listed from the lowest to the highest level. Give every level a short, concrete description and add a one-sentence `instructions`: bare labels such as "Negative" / "Positive" give much less accurate scores.

`expected_score` is 0-based over the rubric, so add 1 to get a 1 to 5 score.

In [ ]:
input_text = "The support agent was friendly and fixed my issue fast." #@param {type:"string"}

rubric = [
    'Very dissatisfied: angry, problem not solved, terrible service',
    'Dissatisfied: slow or unhelpful support, problem partly solved',
    'Neutral: okay, average, nothing special',
    'Satisfied: helpful support, problem solved',
    'Very satisfied: excellent, fast, friendly support, delighted',
]
score_question = decision.ScoreQuestion(
    rubric=rubric,
    instructions='Rate how satisfied the customer is with the support experience.',
)
result = decision_maker.evaluate_score(input_text, score_question)

print(f'Score: {result.expected_score + 1:.2f} / {len(rubric)}')
for level, (description, probability) in enumerate(zip(rubric, result.probabilities), 1):
  print(f'  {level}. {probability:6.1%}  {description}')

Finally, release the model when you are done.

In [ ]:
decision_maker.close()